# Ejercicio 4 — Predicción de abandono y fuga de información
**Universidad El Bosque — Programa de Matemáticas y Estadística — Introducción al Machine Learning — Semestre 2026-2**

Se quiere predecir, en una fecha $t_0$, si un cliente abandonará el servicio en los siguientes 30 días. Antes de ajustar cualquier modelo hay que revisar *cuándo* se mide cada variable, porque un modelo que se ejecuta en $t_0$ solo puede usar información que ya exista en $t_0$.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys

sys.path.append("../src")

# Formato numérico uniforme para las tablas
pd.set_option("display.float_format", lambda v: f"{v:.6f}")

from lista4 import cargar_datos, resumen_logistica

df = cargar_datos("clientes")
print(df.shape)
df.head()

(1200, 7)


,antiguedad_meses,uso_30d_previos,facturas_vencidas_180d,tickets_90d,pago_automatico,ajuste_comercial_30d,abandona_30d
0,66,37.927153,0,1,0,23.039078,0
1,12,46.656223,0,1,1,19.158849,0
2,49,22.990662,0,3,1,27.712278,0
3,49,24.546478,1,0,0,96.997980,1
4,30,64.413222,0,2,1,0.000000,0


## (a) Variables disponibles en $t_0$

La tabla del enunciado indica la ventana de tiempo de cada variable:

| Variable | Ventana | ¿Disponible en $t_0$? |
|---|---|---|
| `antiguedad_meses` | disponible en $t_0$ | Sí |
| `uso_30d_previos` | $[t_0-30,\,t_0]$ | Sí |
| `facturas_vencidas_180d` | $[t_0-180,\,t_0]$ | Sí |
| `tickets_90d` | $[t_0-90,\,t_0]$ | Sí |
| `pago_automatico` | disponible en $t_0$ | Sí |
| `ajuste_comercial_30d` | $[t_0,\,t_0+30]$ | **No**: usa información del futuro |

Todas las ventanas terminan en $t_0$ salvo la de `ajuste_comercial_30d`, que se extiende hasta $t_0+30$, es decir, justo el periodo en el que se observa si el cliente abandona.

In [2]:
variables_disponibles = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico",
]
todas_las_variables = variables_disponibles + ["ajuste_comercial_30d"]
variables_disponibles

['antiguedad_meses',
 'uso_30d_previos',
 'facturas_vencidas_180d',
 'tickets_90d',
 'pago_automatico']

## (b) Ajuste de M1 y M2

M1 usa todas las variables predictoras y M2 solo `variables_disponibles`. Ambos se ajustan con `LogisticRegression(solver='lbfgs', C=1e6, max_iter=10000)`, que equivale a casi no regularizar, de modo que el ajuste se parece al de máxima verosimilitud.

In [3]:
M1 = resumen_logistica(df, todas_las_variables)
M2 = resumen_logistica(df, variables_disponibles)

# Se confirma que lbfgs convergió antes de agotar las iteraciones
print("Iteraciones usadas  M1:", M1["modelo"].n_iter_[0], "| M2:", M2["modelo"].n_iter_[0], "(máximo 10000)")

Iteraciones usadas  M1: 78 | M2: 54 (máximo 10000)


## (c) y (d) Riesgo logístico y errores de clasificación

La clase predicha es 1 cuando $\hat p_i\ge 0.5$. Los errores se cuentan sobre las mismas 1200 observaciones usadas para ajustar.

In [4]:
tabla_4 = pd.DataFrame({
    "Variables usadas":            [len(M1["predictores"]), len(M2["predictores"])],
    "Riesgo empírico logístico":   [M1["riesgo"], M2["riesgo"]],
    "Errores de clasificación":    [M1["errores"], M2["errores"]],
    "Proporción de errores":       [M1["prop_errores"], M2["prop_errores"]],
}, index=["M1 (todas)", "M2 (solo disponibles en t0)"])
tabla_4["Variables usadas"] = tabla_4["Variables usadas"].astype(int)
tabla_4["Errores de clasificación"] = tabla_4["Errores de clasificación"].astype(int)
tabla_4

,Variables usadas,Riesgo empírico logístico,Errores de clasificación,Proporción de errores
M1 (todas),6,0.342485,160,0.133333
M2 (solo disponibles en t0),5,0.571716,336,0.280000


M1 parece claramente mejor: su riesgo logístico baja de ≈ 0.572 a ≈ 0.342 y los errores de clasificación pasan de 336 (28.0 %) a 160 (13.3 %). Pero esa mejora hay que mirarla con cuidado, porque ambos modelos solo se diferencian en una variable. Para ver cuánto pesa esa variable en M1, se pueden comparar los coeficientes:

In [5]:
coef_M1 = pd.Series(M1["modelo"].coef_[0], index=M1["predictores"], name="Coeficiente en M1")
coef_M1.to_frame()

,Coeficiente en M1
antiguedad_meses,-0.005853
uso_30d_previos,-0.007346
facturas_vencidas_180d,0.574446
tickets_90d,0.096497
pago_automatico,-0.501647
ajuste_comercial_30d,0.085644


## (e) ¿Qué modelo puede usarse en $t_0$?

**Respuesta.** El modelo que puede ejecutarse exactamente en $t_0$ es **M2**. La variable que produce la diferencia entre M1 y M2 es **`ajuste_comercial_30d`**, porque se construye con datos entre $t_0$ y $t_0+30$, que todavía no existen cuando se quiere hacer la predicción.

M1 no se puede usar en la práctica, y su aparente mejor desempeño es engañoso: es un caso de **fuga de información (*data leakage*)**. `ajuste_comercial_30d` se mide durante la misma ventana en la que se observa el abandono, así que contiene información posterior a $t_0$ (que puede estar influida por lo que hace el cliente en ese mismo periodo). El modelo "aprende" de información que en el momento real de predicción no tendría, y por eso sus errores bajan de 28 % a 13 %. El desempeño honesto del problema que plantea la empresa es el de M2: riesgo ≈ 0.572 y 336 errores (28.0 %).